# Batch Processing - Gold

We have done a lot of work till now. So well done! Not the last and the most important bit, the **Gold Layer**. Remember our domain question, **What is the total time and total charge dispense for every completed transaction**? It was the exercise which required several joins and window queries. :)  We're here to do it again (the lightweight version) but with the help of the work we did in the **Silver Tier**.

Steps:
* Match `StartTransaction` Requests and Responses
* Join `StopTransaction` Requests and `StartTransaction` Responses, matching on `transaction_id` (left join)
* Find the matching `StartTransaction` Requests (left join)
* Calculate the `total_time` (`withColumn`, `cast`, `maths`)
* Calculate `total_energy` (`withColumn`, `cast`)

**NOTE:** You've already done these exercises before. We absolutely recommend bringing over your answers from that exercise to speed things along (with some minor tweaks), because you already know how to do all of that already! Of course, you're welcome to freshly rewrite your answers to test yourself!

🕒 **Estimated Time For Completion:** 45 minutes

## Set up this Notebook
Before we get started, we need to quickly set up this notebook by installing a helpers, cleaning up your unique working directory (as to not clash with others working in the same space), and setting some variables. Run the following cells using `Shift + Enter`. **NOTE:** that if your cluster shuts down, you will need to re-run the cells in this section.

In [0]:
%pip uninstall -y databricks_helpers exercise_ev_databricks_unit_tests
%pip install git+https://github.com/data-derp/databricks_helpers#egg=databricks_helpers git+https://github.com/data-derp/exercise_ev_databricks_unit_tests#egg=exercise_ev_databricks_unit_tests

In [0]:
from databricks_helpers.databricks_helpers import DataDerpDatabricksHelpers
exercise_name = "batch_processing_gold"
helpers = DataDerpDatabricksHelpers(dbutils, exercise_name)

current_user = helpers.current_user()
working_directory = helpers.working_directory()
print(f"Your current working directory is: {working_directory}")

## This function CLEARS your current working directory. Only run this if you want a fresh start or if it is the first time you're doing this exercise.
helpers.clean_working_directory()

## Read Data from Silver Layer
Let's read the parquet files that we created in the Silver layer!

**NOTE:** normally we'd use the EXACT data and location of the data that was created in the Silver layer but for simplicity and consistent results [of this exercise], we're going to read in a Silver output dataset that has been pre-prepared. Don't worry, it's the same as the output from your exercise (if all of your tests passed)!

In [0]:
meter_values_request_url = "https://github.com/data-derp/exercise-ev-databricks/raw/main/batch-processing-silver/output/MeterValuesRequest/part-00000-tid-468425781006758111-f9d48bc3-3b4c-497e-8e9c-77cf63db98f8-207-1-c000.snappy.parquet"
start_transaction_request_url = "https://github.com/data-derp/exercise-ev-databricks/raw/main/batch-processing-silver/output/StartTransactionRequest/part-00000-tid-9191649339140138460-0a4f58e5-1397-41cc-a6a1-f6756f3332b6-218-1-c000.snappy.parquet"
start_transaction_response_url = "https://github.com/data-derp/exercise-ev-databricks/raw/main/batch-processing-silver/output/StartTransactionResponse/part-00000-tid-5633887168695670016-762a6dfa-619c-412d-b7b8-158ee41df1b2-185-1-c000.snappy.parquet"
stop_transaction_request_url = "https://github.com/data-derp/exercise-ev-databricks/raw/main/batch-processing-silver/output/StopTransactionRequest/part-00000-tid-5108689541678827436-b76f4703-dabf-439a-825d-5343aabc03b6-196-1-c000.snappy.parquet"

meter_values_request_filepath = helpers.download_to_local_dir(meter_values_request_url)
start_transaction_request_filepath = helpers.download_to_local_dir(start_transaction_request_url)
start_transaction_response_filepath = helpers.download_to_local_dir(start_transaction_response_url)
stop_transaction_request_filepath = helpers.download_to_local_dir(stop_transaction_request_url)


In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

def read_parquet(filepath: str) -> DataFrame:
    df = spark.read.parquet(filepath)
    return df
    
start_transaction_request_df = read_parquet(start_transaction_request_filepath)
start_transaction_response_df = read_parquet(start_transaction_response_filepath)
stop_transaction_request_df = read_parquet(stop_transaction_request_filepath)
meter_values_request_df = read_parquet(meter_values_request_filepath)

display(start_transaction_request_df)
display(start_transaction_response_df)
display(stop_transaction_request_df)
display(meter_values_request_df)

### EXERCISE: Match StartTransaction Requests and Responses
In this exercise, match StartTransaction Requests and Responses using a [left join](https://spark.apache.org/docs/3.1.2/api/python/reference/api/pyspark.sql.DataFrame.join.html) on `message_id`.

Target Schema:

<br/>

```
root
 |-- charge_point_id: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- meter_start: integer (nullable = true)
 |-- start_timestamp: timestamp (nullable = true)
```

In [0]:
def match_start_transaction_requests_with_responses(input_df: DataFrame, join_df: DataFrame) -> DataFrame:
### Put your code here.
    join_type: str = "left"

    return input_df.\
        join(join_df, input_df.message_id == join_df.message_id, join_type).\
        select(
            input_df.charge_point_id.alias("charge_point_id"), 
            input_df.transaction_id.alias("transaction_id"), 
            join_df.meter_start.alias("meter_start"), 
            join_df.timestamp.alias("start_timestamp")
        )
    start_transaction_response_df
display(start_transaction_response_df.transform(match_start_transaction_requests_with_responses, start_transaction_request_df))

#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.batch_processing_gold import test_match_start_transaction_requests_with_responses_unit

test_match_start_transaction_requests_with_responses_unit(spark, match_start_transaction_requests_with_responses)

#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.batch_processing_gold import test_match_start_transaction_requests_with_responses_e2e

test_match_start_transaction_requests_with_responses_e2e(start_transaction_response_df.transform(match_start_transaction_requests_with_responses, start_transaction_request_df), display)



### EXERCISE: Join StopTransaction Requests and StartTransaction Responses
In this exercise, [left join](https://spark.apache.org/docs/3.1.2/api/python/reference/api/pyspark.sql.DataFrame.join.html) `StopTransaction` Requests and the newly joined `StartTransaction` Request/Response DataFrame (from the previous exercise), matching on `transaction_id` (left join).

Target Schema:

<br/>

```
root
 |-- charge_point_id: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- meter_start: integer (nullable = true)
 |-- meter_stop: integer (nullable = true)
 |-- start_timestamp: timestamp (nullable = true)
 |-- stop_timestamp: timestamp (nullable = true)
 ```

In [0]:
def join_with_start_transaction_responses(input_df: DataFrame, join_df: DataFrame) -> DataFrame:
### Put your code here.
    join_type: str = "left"

    return input_df. \
    join(join_df, input_df.transaction_id == join_df.transaction_id, join_type). \
    select(
        join_df.charge_point_id, 
        join_df.transaction_id, 
        join_df.meter_start, 
        input_df.meter_stop.alias("meter_stop"), 
        join_df.start_timestamp, 
        input_df.timestamp.alias("stop_timestamp")
    )

    
display(stop_transaction_request_df.\
    transform(
        join_with_start_transaction_responses, 
        start_transaction_response_df.\
            transform(match_start_transaction_requests_with_responses, start_transaction_request_df)
    )
)

#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.batch_processing_gold import test_join_with_start_transaction_responses_unit

test_join_with_start_transaction_responses_unit(spark, join_with_start_transaction_responses)


#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.batch_processing_gold import test_join_with_start_transaction_responses_e2e

test_join_with_start_transaction_responses_e2e(stop_transaction_request_df.\
    transform(
        join_with_start_transaction_responses, 
        start_transaction_response_df.\
            transform(match_start_transaction_requests_with_responses, start_transaction_request_df)
    ), display)

### EXERCISE: Calculate the total_time
Using Pyspark functions [`withColumn`](https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.DataFrame.withColumn.html) and [`cast`](https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.Column.cast.html?highlight=cast#pyspark.sql.Column.cast) and little creative maths, calculate the total charging time (`stop_timestamp - start_timestamp`) in hours (two decimal places).

Target Schema:

<br/>

```
root
 |-- charge_point_id: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- meter_start: integer (nullable = true)
 |-- meter_stop: integer (nullable = true)
 |-- start_timestamp: string (nullable = true)
 |-- stop_timestamp: string (nullable = true)
 |-- total_time: double (nullable = true)
```

In [0]:
from pyspark.sql.functions import col, round
from pyspark.sql.types import DoubleType

def calculate_total_time(input_df: DataFrame) -> DataFrame:
    seconds_in_one_hour = 3600
    stop_timestamp_column_name: str = None
    start_timestamp_column_name: str = None
### Put your code here.
    stop_timestamp_column_name: str = "stop_timestamp"
    start_timestamp_column_name: str = "start_timestamp"

    return input_df. \
        withColumn("total_time", col(stop_timestamp_column_name).cast("long")/seconds_in_one_hour - col(start_timestamp_column_name).cast("long")/seconds_in_one_hour). \
        withColumn("total_time", round(col("total_time").cast(DoubleType()),2))
    
display(stop_transaction_request_df.\
    transform(
        join_with_start_transaction_responses, 
        start_transaction_response_df.\
            transform(match_start_transaction_requests_with_responses, start_transaction_request_df)
    ).\
    transform(calculate_total_time)
)



#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.batch_processing_gold import test_calculate_total_time_unit

test_calculate_total_time_unit(spark, calculate_total_time)

#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.batch_processing_gold import test_calculate_total_time_hours_e2e

test_calculate_total_time_hours_e2e(
    stop_transaction_request_df.\
    transform(
        join_with_start_transaction_responses, 
        start_transaction_response_df.\
            transform(match_start_transaction_requests_with_responses, start_transaction_request_df)
    ).\
    transform(calculate_total_time), display
)


### EXERCISE: Calculate total_energy
Calculate total_energy (`withColumn`, `cast`)
Using [`withColumn`](https://spark.apache.org/docs/3.1.2/api/python/reference/api/pyspark.sql.DataFrame.withColumn.html?highlight=withcolumn#pyspark.sql.DataFrame.withColumn) and [`cast`](https://spark.apache.org/docs/3.1.2/api/python/reference/api/pyspark.sql.Column.cast.html?highlight=cast#pyspark.sql.Column.cast), calculate the total energy by subtracting `meter_stop` from `meter_start`, converting that value from Wh (Watt-hours) to kWh (kilo-Watt-hours), and rounding to the nearest 2 decimal points.

Target Schema:

<br/>

```
root
 |-- charge_point_id: string (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- meter_start: integer (nullable = true)
 |-- meter_stop: integer (nullable = true)
 |-- start_timestamp: timestamp (nullable = true)
 |-- stop_timestamp: timestamp (nullable = true)
 |-- total_time: double (nullable = true)
 |-- total_energy: double (nullable = true)
 ```

 **HINT:** Wh -> kWh = divide by 1000

In [0]:
def calculate_total_energy(input_df: DataFrame) -> DataFrame:
    meter_stop_column_name: str = None
    meter_start_column_name: str = None
### Put your code here.
    meter_stop_column_name: str = "meter_stop"
    meter_start_column_name: str = "meter_start"

    return input_df \
        .withColumn("total_energy", (col(meter_stop_column_name) - col(meter_start_column_name))/1000) \
        .withColumn("total_energy", round(col("total_energy").cast(DoubleType()),2))
    

result_df = stop_transaction_request_df.\
    transform(
        join_with_start_transaction_responses, 
        start_transaction_response_df.\
            transform(match_start_transaction_requests_with_responses, start_transaction_request_df)
    ).\
    transform(calculate_total_time).\
    transform(calculate_total_energy)

display(result_df)

#### Unit Test

In [0]:
from exercise_ev_databricks_unit_tests.batch_processing_gold import test_calculate_total_energy_unit

test_calculate_total_energy_unit(spark, calculate_total_energy)

#### E2E Test

In [0]:
from exercise_ev_databricks_unit_tests.batch_processing_gold import test_calculate_total_energy_e2e

test_calculate_total_energy_e2e(stop_transaction_request_df.\
    transform(
        join_with_start_transaction_responses, 
        start_transaction_response_df.\
            transform(match_start_transaction_requests_with_responses, start_transaction_request_df)
    ).\
    transform(calculate_total_time).\
    transform(calculate_total_energy), display)

So we have calculated both the total time and total charge dispensed (total energy). This marks the end of this exercise. Lets do the clean up below

In [0]:
from pathlib import Path

helpers.clean_working_directory()

# we need the data product in the next workbook
output_directory = Path(f"{working_directory}/../products/total_energy").resolve()
result_df.\
    write.\
    parquet(str(output_directory), mode='overwrite')

print("Total energy is written to: ", output_directory)

## Wrap-up

Congratulations, you have applied the medallion or multi-hop artechitecture using the bronze, silver and gold layer. It's handy to batch processing. Notice that you've read/written Parquet files through the process.

Let's move on in this great learning journey to the next module. You are doing a great job!

| Previous Topic                                                          | Next Topic                                                        |
|-------------------------------------------------------------------------|-------------------------------------------------------------------|
| <a href="$./5.4 Exercise - Silver" target="_self">Exercise - Silver</a> | <a href="$./5.6 Data-Quality" target="_self">Data Quality</a>     |

&copy; 2024 Thoughtworks. All rights reserved.<br/>